In [ ]:
import fitz
import os
ficha = "3204759"
def convertir_pdf_a_imagenes_consecutivas_jpg(archivo_pdf, ruta_salida, dpi=300):
    """Convierte un PDF a imágenes JPG con nombres consecutivos (imagen1.jpg, imagen2.jpg, etc.)."""
    try:
        pdf_documento = fitz.open(archivo_pdf)
        os.makedirs(ruta_salida, exist_ok=True)

        for num_pagina in range(len(pdf_documento)):
            pagina = pdf_documento.load_page(num_pagina)
            pix = pagina.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
            nombre_archivo_salida = f"{ruta_salida}/imagen{num_pagina + 1}.jpg"
            pix.save(nombre_archivo_salida, "JPEG")

        pdf_documento.close()
        print(f"PDF convertido a imágenes JPG en '{ruta_salida}' con nombres consecutivos.")
    except Exception as e:
        print(f"Ocurrió un error: {e}")

# Ejemplo de uso:
convertir_pdf_a_imagenes_consecutivas_jpg(f"./fichasatrascribir/{ficha}.pdf", "./imagenes_jpg")

In [ ]:
import base64
import os
import datetime
import shutil
from mistralai import Mistral
from dotenv import load_dotenv
import json
import time
import pandas as pd
import numpy as np

# --- INICIO: Configuración ---
SOURCE_IMAGE_FOLDER = "./imagenes_jpg" # Carpeta donde están las imágenes originales
ALLOWED_EXTENSIONS = ('.jpg', '.jpeg', '.png') # Extensiones de imagen a procesar (en minúsculas)
# --- FIN: Configuración ---

# --- INICIO: Listas para almacenar resultados (SIN CAMBIOS) ---
lista_de_datos = []
lista_errores_primarios = []
lista_errores_reintento_fallido = []
# --- FIN: Listas para almacenar resultados ---

load_dotenv()

# --- INICIO: Función real para codificar imágenes (SIN CAMBIOS) ---
def encode_image(image_path):
    """Codifica la imagen local a base64."""
    try:
        with open(image_path, "rb") as image_file:
            image_bytes = image_file.read()
            base64_bytes = base64.b64encode(image_bytes)
            base64_string = base64_bytes.decode('utf-8')
            return base64_string
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {image_path}")
        return None
    except Exception as e:
        print(f"Error al codificar la imagen '{image_path}': {e}")
        return None
# --- FIN: Función real para codificar imágenes ---

# --- INICIO: Crear Directorio de Destino para Procesados (SIN CAMBIOS) ---
now = datetime.datetime.now()
timestamp_str = now.strftime("%Y-%m-%d_%H-%M-%S")
destination_folder_name = f"{timestamp_str}_procesados"
destination_dir_path = os.path.join(".", destination_folder_name)
try:
    os.makedirs(destination_dir_path, exist_ok=True)
    print(f"Directorio de destino para imágenes procesadas: '{destination_dir_path}'")
except OSError as e:
    print(f"Error crítico al crear el directorio de destino '{destination_dir_path}': {e}")
    destination_dir_path = None
# --- FIN: Crear Directorio de Destino ---

# --- INICIO: Escanear, Contar, Renombrar y Listar Archivos ---
print(f"\n--- Preparando archivos en: '{SOURCE_IMAGE_FOLDER}' ---")
files_to_process = [] # Lista con las NUEVAS rutas de archivo a procesar
carchivos = 0         # Contador total de archivos de imagen encontrados
original_image_files_map = {} # Para mapear nombre original a nuevo nombre (opcional, para info)

try:
    # 1. Listar y Filtrar Imágenes Válidas
    all_entries = os.listdir(SOURCE_IMAGE_FOLDER)
    valid_image_filenames = []
    for entry in all_entries:
        entry_path = os.path.join(SOURCE_IMAGE_FOLDER, entry)
        # Comprobar si es archivo y tiene extensión permitida (ignora mayúsculas/minúsculas)
        if os.path.isfile(entry_path) and entry.lower().endswith(ALLOWED_EXTENSIONS):
            valid_image_filenames.append(entry)

    # 2. Contar
    carchivos = len(valid_image_filenames)
    print(f"Se encontraron {carchivos} archivos de imagen válidos.")

    # Ordenar para un renombrado consistente (alfabético)
    valid_image_filenames.sort()

    # 3. Renombrar y Crear Lista de Procesamiento
    consecutive_counter = 1
    print("Renombrando archivos a formato consecutivo...")
    for original_filename in valid_image_filenames:
        try:
            old_path = os.path.join(SOURCE_IMAGE_FOLDER, original_filename)
            # Obtener extensión (ej: '.jpg')
            _, file_extension = os.path.splitext(original_filename)
            file_extension = file_extension.lower() # Asegurar extensión en minúsculas

            # Crear nuevo nombre y ruta
            new_filename = f"{consecutive_counter}{file_extension}"
            new_path = os.path.join(SOURCE_IMAGE_FOLDER, new_filename)

            # Evitar renombrar si ya tiene el nombre correcto
            if old_path == new_path:
                print(f"  - Archivo '{original_filename}' ya tiene el nombre correcto. Añadiendo a la lista.")
                if new_path not in files_to_process: # Evitar duplicados si ya existía
                     files_to_process.append(new_path)
                original_image_files_map[original_filename] = new_filename # Mapeo
                consecutive_counter += 1
                continue

            # Comprobar si el nombre nuevo ya existe (conflicto)
            if os.path.exists(new_path):
                print(f"  - ¡Advertencia! El nombre de archivo destino '{new_filename}' ya existe.")
                print(f"    Se omitirá el renombrado de '{original_filename}'.")
                print(f"    Si '{new_filename}' es un archivo válido, será procesado si aún no está en la lista.")
                # Podrías añadir new_path a files_to_process si es válido y no está ya
                if os.path.isfile(new_path) and new_path.lower().endswith(ALLOWED_EXTENSIONS) and new_path not in files_to_process:
                    files_to_process.append(new_path)
                continue # Saltar el renombrado conflictivo

            # Renombrar
            os.rename(old_path, new_path)
            print(f"  - Renombrado: '{original_filename}' -> '{new_filename}'")
            files_to_process.append(new_path) # Añadir la NUEVA ruta a la lista
            original_image_files_map[original_filename] = new_filename # Mapeo
            consecutive_counter += 1

        except OSError as e_rename:
            print(f"  - ¡ERROR! No se pudo renombrar '{original_filename}' a '{new_filename}': {e_rename}")
            # Registrar error si se desea
            lista_errores_primarios.append({
                "imagen": original_filename, # Nombre original
                "razon": "Error al Renombrar Archivo",
                "detalle": f"{type(e_rename).__name__}: {e_rename}"
            })

except FileNotFoundError:
    print(f"¡ERROR CRÍTICO! El directorio fuente '{SOURCE_IMAGE_FOLDER}' no fue encontrado.")
    carchivos = 0 # Asegura que el bucle no se ejecute
    # Considerar salir del script: exit()
except Exception as e_scan:
    print(f"¡ERROR CRÍTICO! Ocurrió un problema al escanear/renombrar archivos en '{SOURCE_IMAGE_FOLDER}': {e_scan}")
    carchivos = 0 # Asegura que el bucle no se ejecute
    # Considerar salir del script: exit()

print(f"Se procesarán {len(files_to_process)} archivos.")
# Puedes imprimir el mapeo si es útil:
# print("Mapeo de nombres original -> nuevo:", original_image_files_map)
# --- FIN: Escanear, Contar, Renombrar y Listar Archivos ---


# --- INICIO: Bucle Principal - Iterar sobre la LISTA de archivos renombrados ---
print("\n--- Iniciando Procesamiento de Imágenes ---")
if not files_to_process:
    print("No hay archivos válidos para procesar.")

# Iterar sobre la lista de rutas completas obtenida previamente
for local_image_path in files_to_process:

    # Pausa breve entre procesamientos
    time.sleep(1)

    # --- Variables y chequeo API Key ---
    api_key = os.getenv("MISTRAL_API_KEY")
    if not api_key:
        print("Error Fatal: MISTRAL_API_KEY no configurada. Deteniendo procesamiento.")
        # Registrar error de configuración si no se hizo antes
        if not any(err['razon'] == "Configuración" for err in lista_errores_primarios):
             lista_errores_primarios.append({ "imagen": "N/A", "razon": "Configuración", "detalle": "MISTRAL_API_KEY no encontrada"})
        break # Salir del bucle for si falta la API key

    model = "mistral-small-latest"
    text_prompt = "dame en formato json el nombre la identificacion, la identificicacion debe ser un numero sin separacion de puntos"

    # 'local_image_path' ya es la ruta completa del archivo (renombrado)
    print(f"\nProcesando imagen: {local_image_path}") # Muestra la ruta renombrada

    # --- 1. Intento de Codificación ---
    # Usa directamente la ruta de la lista
    base64_image_data = encode_image(local_image_path)
    if base64_image_data is None:
        # Registrar error usando local_image_path
        datos_error = { "imagen": local_image_path, "razon": "Error de Codificación", "detalle": "Fallo al leer/codificar."}
        lista_errores_primarios.append(datos_error)
        print(f"Error (No reintentable) registrado para {local_image_path}: {datos_error['razon']}")
        continue

    # --- Preparación API ---
    # Determinar tipo de imagen desde la extensión del archivo renombrado
    _, img_ext = os.path.splitext(local_image_path)
    image_type = img_ext.lower().replace('.', '')
    # Ajustar para API ('jpg' -> 'jpeg')
    if image_type == 'jpg': image_type = 'jpeg'
    # Lista simple de tipos comunes, puedes expandir o ajustar
    supported_types = ['jpeg', 'png', 'gif', 'webp']
    if image_type not in supported_types:
        print(f"  - Advertencia: Tipo de imagen '{image_type}' desconocido. Usando 'jpeg' por defecto.")
        image_type = 'jpeg'

    image_input_for_api = f"data:image/{image_type};base64,{base64_image_data}"
    messages = [
        {"role": "user", "content": [
            {"type": "text", "text": text_prompt},
            {"type": "image_url", "image_url": image_input_for_api}
        ]}
    ]

    # --- 2. Intento de Llamada a API con Reintento ---
    # La lógica interna es la misma, pero usa 'local_image_path' en logs/errores
    max_attempts = 2
    attempt = 0
    interpretation_result = None
    api_call_successful = False
    last_api_error = None

    while attempt < max_attempts:
        attempt += 1
        try:
            client = Mistral(api_key=api_key)
            # Usar basename para el print para que sea más corto
            print(f"Intento {attempt}/{max_attempts}: Enviando solicitud para '{os.path.basename(local_image_path)}'...")
            chat_response = client.chat.complete(model=model, messages=messages)
            if not chat_response.choices:
                raise ValueError(f"Respuesta API inválida (sin 'choices')")
            interpretation_result = chat_response.choices[0].message.content
            api_call_successful = True
            print(f"Intento {attempt} exitoso.")
            break
        except Exception as e_api:
            last_api_error = e_api
            print(f"\nError durante el intento {attempt} de API para '{os.path.basename(local_image_path)}': {e_api}")
            if attempt < max_attempts:
                print(f"Reintentando en {2 * attempt} segundos...")
                time.sleep(2 * attempt)
            else:
                print(f"La API falló para '{os.path.basename(local_image_path)}' después de {max_attempts} intentos.")
                datos_error = {
                    "imagen": local_image_path, # Ruta renombrada
                    "razon": f"Error de API tras {max_attempts} intentos",
                    "detalle": f"{type(last_api_error).__name__}: {last_api_error}"
                }
                lista_errores_reintento_fallido.append(datos_error)
                print(f"Error de API (tras reintento) registrado para {local_image_path}")

    if not api_call_successful:
        continue

    # --- API call fue exitosa ---
    print("\n--- Respuesta del Modelo ---")
    print(interpretation_result)
    print("-" * 30)

    # --- 4. Intento de Procesamiento JSON y MOVER ARCHIVO ---
    # La lógica interna es la misma, usa 'local_image_path'
    try:
        json_string = interpretation_result.split('```json')[1].split('```')[0].strip()
        datos = json.loads(json_string)

        # Éxito: Añadir datos
        lista_de_datos.append(datos)
        print(f"Datos extraídos de '{os.path.basename(local_image_path)}' y añadidos a la lista.")

        # Mover archivo (usa local_image_path que es la ruta renombrada)
        if destination_dir_path and os.path.exists(local_image_path):
            try:
                file_name = os.path.basename(local_image_path) # Ya es el nombre consecutivo
                destination_file_path = os.path.join(destination_dir_path, file_name)
                shutil.move(local_image_path, destination_file_path)
                print(f"Imagen '{file_name}' movida exitosamente a '{destination_dir_path}'")
            except Exception as e_move:
                print(f"¡ERROR! No se pudo mover '{local_image_path}': {e_move}")
                lista_errores_primarios.append({
                    "imagen": local_image_path,
                    "razon": "Error al Mover Archivo Procesado",
                    "detalle": f"{type(e_move).__name__}: {e_move}"
                })
        elif not os.path.exists(local_image_path):
             print(f"Advertencia: El archivo '{local_image_path}' no existe, no se puede mover (¿fue movido o eliminado?).")

    except (json.JSONDecodeError, KeyError, IndexError, AttributeError, ValueError) as e_json:
        # Error JSON (usa local_image_path en logs)
        print(f"Error al procesar el JSON de '{os.path.basename(local_image_path)}': {e_json}")
        # ... (resto del manejo de error JSON como antes, usando local_image_path) ...
        datos_error = {
            "imagen": local_image_path,
            "razon": "Error Procesamiento JSON",
            "detalle": f"{type(e_json).__name__}: {e_json}"
        }
        lista_errores_primarios.append(datos_error)
        print(f"Error (No reintentable) registrado para {local_image_path}: {datos_error['razon']}")


# --- INICIO: Creación de DataFrames Finales (SIN CAMBIOS EN LÓGICA) ---
# ... (Impresión de los 3 DataFrames como antes) ...
print("\n" + "="*60)
print("PROCESO COMPLETADO - RESUMEN FINAL")
print("="*60 + "\n")
# ... (Código de impresión de DataFrames) ...

# --- INICIO DEL CÓDIGO ---
# Se asume que la variable 'lista_de_datos' ya existe y contiene los datos.
# NO se redefine aquí, se usa la existente en tu entorno.

print("Procesando la variable 'lista_de_datos' existente...")

try:
    # 1. Convertir la lista (preexistente) a un DataFrame inicial
    #    Esto funcionará siempre que 'lista_de_datos' sea una lista de diccionarios.
    df = pd.DataFrame(lista_de_datos)
    
    print("\nDataFrame Inicial creado desde 'lista_de_datos':")
    # Mostrar las primeras filas y la información general puede ser útil
    print(df.head()) 
    print(df.info())
    print("-" * 30)

    # 2. Verificar SI la columna 'apellido' existe en el DataFrame
    if 'apellido' in df.columns:
        print("Columna 'apellido' encontrada. Iniciando proceso de unificación...")

        # 3. Identificar filas donde 'apellido' NO es nulo (para la concatenación)
        indices_con_apellido = df['apellido'].notna()

        # 4. Concatenar 'nombre' y 'apellido' SOLO en esas filas
        #    Usamos .loc para asegurar que modificamos el DataFrame original.
        #    Usamos .fillna('') en ambas columnas antes de sumar para evitar errores
        #    si 'nombre' fuera nulo donde 'apellido' sí existe.
        df.loc[indices_con_apellido, 'nombre'] = df.loc[indices_con_apellido, 'nombre'].fillna('') + ' ' + df.loc[indices_con_apellido, 'apellido'].fillna('')

        # 5. Eliminar la columna 'apellido' ya que se integró a 'nombre'
        df.drop(columns=['apellido'], inplace=True)
        print("Unificación completada. Columna 'apellido' eliminada.")

    else:
        # Mensaje si la columna 'apellido' no se encontró
        print("La columna 'apellido' NO existe en el DataFrame creado desde 'lista_de_datos'.")
        print("No se realizará la unificación de nombre y apellido ni se eliminará la columna.")

    # 6. Mostrar el DataFrame final (ya sea modificado o no)
    print("\nDataFrame Final procesado:")
    print(df.head()) # Mostrar las primeras filas del resultado
    print(df.info()) # Mostrar información final de columnas y tipos

except NameError:
    # Error si la variable 'lista_de_datos' no existe realmente
    print("¡Error! La variable 'lista_de_datos' no está definida.")
    print("Por favor, asegúrate de que 'lista_de_datos' exista y contenga los datos antes de ejecutar este código.")
except Exception as e:
    # Captura de otros posibles errores durante la creación o manipulación del DataFrame
    print(f"¡Error inesperado durante el procesamiento del DataFrame!")
    print(f"Detalle del error: {e}")
    # Si el df se creó parcialmente, mostrarlo puede ayudar a diagnosticar
    if 'df' in locals():
        print("\nEstado parcial del DataFrame antes del error:")
        print(df.head())
        print(df.info())

# --- FIN DEL CÓDIGO ---
# --- INICIO DEL CÓDIGO ---
# Se asume que la variable 'lista_de_datos' ya existe y contiene los datos.
# NO se redefine aquí, se usa la existente en tu entorno.

print("Procesando la variable 'lista_de_datos' existente...")

try:
    # 1. Convertir la lista (preexistente) a un DataFrame inicial
    #    Esto funcionará siempre que 'lista_de_datos' sea una lista de diccionarios.
    df = pd.DataFrame(lista_de_datos)
    
    print("\nDataFrame Inicial creado desde 'lista_de_datos':")
    # Mostrar las primeras filas y la información general puede ser útil
    print(df.head()) 
    print(df.info())
    print("-" * 30)

    # 2. Verificar SI la columna 'apellido' existe en el DataFrame
    if 'apellido' in df.columns:
        print("Columna 'apellido' encontrada. Iniciando proceso de unificación...")

        # 3. Identificar filas donde 'apellido' NO es nulo (para la concatenación)
        indices_con_apellido = df['apellido'].notna()

        # 4. Concatenar 'nombre' y 'apellido' SOLO en esas filas
        #    Usamos .loc para asegurar que modificamos el DataFrame original.
        #    Usamos .fillna('') en ambas columnas antes de sumar para evitar errores
        #    si 'nombre' fuera nulo donde 'apellido' sí existe.
        df.loc[indices_con_apellido, 'nombre'] = df.loc[indices_con_apellido, 'nombre'].fillna('') + ' ' + df.loc[indices_con_apellido, 'apellido'].fillna('')

        # 5. Eliminar la columna 'apellido' ya que se integró a 'nombre'
        df.drop(columns=['apellido'], inplace=True)
        print("Unificación completada. Columna 'apellido' eliminada.")

    else:
        # Mensaje si la columna 'apellido' no se encontró
        print("La columna 'apellido' NO existe en el DataFrame creado desde 'lista_de_datos'.")
        print("No se realizará la unificación de nombre y apellido ni se eliminará la columna.")

    # 6. Mostrar el DataFrame final (ya sea modificado o no)
    print("\nDataFrame Final procesado:")
    print(df.head()) # Mostrar las primeras filas del resultado
    print(df.info()) # Mostrar información final de columnas y tipos

except NameError:
    # Error si la variable 'lista_de_datos' no existe realmente
    print("¡Error! La variable 'lista_de_datos' no está definida.")
    print("Por favor, asegúrate de que 'lista_de_datos' exista y contenga los datos antes de ejecutar este código.")
except Exception as e:
    # Captura de otros posibles errores durante la creación o manipulación del DataFrame
    print(f"¡Error inesperado durante el procesamiento del DataFrame!")
    print(f"Detalle del error: {e}")
    # Si el df se creó parcialmente, mostrarlo puede ayudar a diagnosticar
    if 'df' in locals():
        print("\nEstado parcial del DataFrame antes del error:")
        print(df.head())
        print(df.info())

# --- FIN DEL CÓDIGO ---
print("Preparando datos de la lista original...")
df_list = df.copy()
df_list['identificacion'] = df_list['identificacion'].astype(str).str.strip()
df_list['nombre'] = df_list['nombre'].astype(str).str.strip()
df_list['nombre_norm'] = df_list['nombre'].str.lower()
df_list.dropna(subset=['identificacion'], inplace=True)
df_list = df_list[df_list['identificacion'] != '']
print(f"Datos de lista original preparados: {len(df_list)} registros válidos.")

# --- PASO 2: Cargar Datos del Excel (df_excel) ---
excel_folder = "fichasatrascribir"
excel_filename = f"{ficha}.xls"
excel_file_path = os.path.join(excel_folder, excel_filename)
# Usaremos v3 para este nuevo informe con limpieza de prefijo
output_excel_path = f"informe_comparacion_{ficha}.xlsx"

print(f"Intentando cargar datos desde: {excel_file_path}")
try:
    df_excel = pd.read_excel(
        excel_file_path,
        header=5,
        usecols="A,B",
        engine='xlrd'
    )
    df_excel.columns = ['identificacion', 'nombre']

    # Limpieza de datos del Excel (MODIFICACIÓN AQUÍ):
    df_excel.dropna(subset=['identificacion'], inplace=True) # Requiere ID

    # 1. Convertir toda la columna a texto para poder manipularla
    df_excel['identificacion'] = df_excel['identificacion'].astype(str)

    # 2. Eliminar prefijos como "CC - ":
    #    - Se divide el texto por " - ". Si existe, toma la última parte (el número).
    #    - Si " - " no existe, simplemente devuelve el texto original.
    #    - Se usa apply con lambda para manejar ambos casos de forma segura.
    print("Aplicando limpieza de prefijos a las identificaciones del Excel...")
    df_excel['identificacion'] = df_excel['identificacion'].apply(
        lambda x: x.split(' - ')[-1] if pd.notna(x) and ' - ' in x else x
    )

    # 3. Eliminar '.0' residual (si Excel lo interpretó como número) y quitar espacios
    df_excel['identificacion'] = df_excel['identificacion'].str.replace(r'\.0$', '', regex=True).str.strip()

    # Limpieza del nombre (igual que antes)
    df_excel['nombre'] = df_excel['nombre'].fillna('').astype(str).str.strip()
    df_excel['nombre_norm'] = df_excel['nombre'].str.lower()

    # Filtrar IDs vacíos después de toda la limpieza
    df_excel = df_excel[df_excel['identificacion'] != '']

    print(f"Datos cargados y limpiados desde Excel (con limpieza de prefijo): {len(df_excel)} registros válidos.")

except FileNotFoundError:
    print(f"Error Crítico: El archivo no se encontró en la ruta: {excel_file_path}")
    exit()
except ImportError:
    print("Error Crítico: Necesitas instalar la librería 'xlrd'. Ejecuta: pip install xlrd")
    exit()
except Exception as e:
    print(f"Error Crítico al leer o procesar el archivo Excel: {e}")
    exit()

# --- PASO 3: Fusionar (Merge) los Datos ---
# (Sin cambios aquí, usa las IDs ya limpias)
df_merged = pd.merge(
    df_list[['identificacion', 'nombre', 'nombre_norm']],
    df_excel[['identificacion', 'nombre', 'nombre_norm']],
    on='identificacion',
    how='outer',
    suffixes=('_lista', '_excel'),
    indicator=True
)
print("Datos fusionados. Iniciando análisis con IDs limpias y comparación case-insensitive...")

# --- PASO 4: Identificar Coincidencias (Paridad) y Discrepancias ---
# (Sin cambios en esta lógica, opera sobre los datos fusionados y limpios)

# 4.1: Coincidencias (Paridad)
matches_condition = (df_merged['_merge'] == 'both') & (df_merged['nombre_norm_lista'] == df_merged['nombre_norm_excel'])
df_matches = df_merged.loc[matches_condition, ['identificacion', 'nombre_lista']].rename(columns={'nombre_lista': 'nombre'})
df_matches.drop_duplicates(subset=['identificacion'], keep='first', inplace=True)
print(f"Registros con paridad (Misma ID limpia, Mismo Nombre case-insensitive): {len(df_matches)}")

# 4.2: Discrepancias (Sin Paridad)
# Tipo 1: Diferencia de Nombre
name_mismatch_condition = (df_merged['_merge'] == 'both') & (df_merged['nombre_norm_lista'] != df_merged['nombre_norm_excel'])
df_name_mismatch = df_merged.loc[name_mismatch_condition, ['identificacion', 'nombre_lista', 'nombre_excel']].copy()
df_name_mismatch['tipo_discrepancia'] = 'Diferencia de Nombre (Misma ID)'
df_name_mismatch.rename(columns={'nombre_lista': 'nombre_en_lista', 'nombre_excel': 'nombre_en_excel'}, inplace=True)
df_name_mismatch.drop_duplicates(subset=['identificacion'], keep='first', inplace=True)
print(f"Discrepancias - Diferencia de nombre: {len(df_name_mismatch)}")

# Tipo 2: Solo en Lista Original
only_list_condition = df_merged['_merge'] == 'left_only'
df_only_list = df_merged.loc[only_list_condition, ['identificacion', 'nombre_lista']].copy()
df_only_list['tipo_discrepancia'] = 'Solo en Lista Original'
df_only_list['nombre_en_lista'] = df_only_list['nombre_lista']
df_only_list['nombre_en_excel'] = None
df_only_list.drop_duplicates(subset=['identificacion'], keep='first', inplace=True)
print(f"Discrepancias - Solo en lista original: {len(df_only_list)}")

# Tipo 3: Solo en Excel
only_excel_condition = df_merged['_merge'] == 'right_only'
df_only_excel = df_merged.loc[only_excel_condition, ['identificacion', 'nombre_excel']].copy()
df_only_excel['tipo_discrepancia'] = 'Solo en Excel'
df_only_excel['nombre_en_lista'] = None
df_only_excel['nombre_en_excel'] = df_only_excel['nombre_excel']
df_only_excel.drop_duplicates(subset=['identificacion'], keep='first', inplace=True)
print(f"Discrepancias - Solo en Excel: {len(df_only_excel)}")

# 4.3: Combinar todas las Discrepancias
df_discrepancies = pd.concat([
    df_name_mismatch[['identificacion', 'nombre_en_lista', 'nombre_en_excel', 'tipo_discrepancia']],
    df_only_list[['identificacion', 'nombre_en_lista', 'nombre_en_excel', 'tipo_discrepancia']],
    df_only_excel[['identificacion', 'nombre_en_lista', 'nombre_en_excel', 'tipo_discrepancia']]
], ignore_index=True)
df_discrepancies = df_discrepancies[['identificacion', 'tipo_discrepancia', 'nombre_en_lista', 'nombre_en_excel']]
print(f"Total de registros con discrepancias (combinados): {len(df_discrepancies)}")


# --- PASO 5: Generar el Informe Excel ---
print(f"Generando informe Excel v3 en: {output_excel_path}")
try:
    with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
        df_matches.to_excel(writer, sheet_name='Datos con Paridad', index=False)
        df_discrepancies.to_excel(writer, sheet_name='Datos sin Paridad (Discrepancias)', index=False)

    print("="*30)
    print(f"¡Éxito! El informe de comparación v3 ha sido generado en:")
    print(os.path.abspath(output_excel_path))
    print("Se aplicó limpieza de prefijos en IDs y comparación case-insensitive para nombres.")
    print("="*30)

except ImportError:
    print("Error Crítico: Necesitas instalar la librería 'openpyxl'. Ejecuta: pip install openpyxl")
except Exception as e:
    print(f"Error Crítico al escribir el archivo Excel de salida: {e}")